In [ ]:
pip install --upgrade langchain_openai

In [3]:
import os
import shutil
from pathlib import Path
from langchain_openai import ChatOpenAI

OPENAI_API_KEY = 'Your key'
MODEL_NAME = "gpt-4o-mini"

# Keep margin for response output
MAX_INPUT_TOKENS = 90000
MAX_OUTPUT_TOKENS = 3000

def rough_token_estimate(text: str) -> int:
    return max(1, len(text) // 4)

def read_pipeline_files(directory, start_idx=None, end_idx=None):
    """
    Read pipeline files from a selected index range.
    Example:
      start_idx=1000, end_idx=2000  -> reads files[1000:2000]
    """
    all_filenames = sorted([
        filename for filename in os.listdir(directory)
        if filename.endswith(".pipeline") or filename.endswith(".Pipeline")
    ])

    selected_filenames = all_filenames[start_idx:end_idx]

    pipeline_contents = []
    for filename in selected_filenames:
        path = os.path.join(directory, filename)
        with open(path, "r", encoding="utf-8", errors="ignore") as file:
            pipeline_contents.append({
                "filename": filename,
                "path": path,
                "content": file.read()
            })

    return pipeline_contents

def build_base_prompt() -> str:
    return """
You are analyzing a dataset of actual DevOps YAML pipeline files.

Your task is to identify recurring STRUCTURAL PATTERNS from the provided pipeline file contents.

Definition of a structural pattern:
A structural pattern is a repeated YAML configuration and execution settings that appears across multiple files.

Important grounding rules:
- Use ONLY the provided pipeline file contents as evidence.
- Do NOT invent fields that are not supported by the input.
- Do NOT rely on generic prior knowledge Devops Pipelines unless it is directly supported by the provided files.
- If a pattern is inferred by abstraction, clearly say so.
- If evidence is weak, mark that pattern as uncertain.
- Do NOT include illustrative example values unless they actually occur in the input.

What to look for:
- Repeated Devops structure and functions across CI/CD pipeline files
- Repeated combinations of fields, not just single keys
- Repeated nested arrangements and co-occurring elements

For each discovered pattern, provide:
1. Top 3 most commonly occurring pattern of yaml structure in given CI/CD pipeline files, and output in format of yaml structure.

Now analyze the following pipeline file contents:

""".strip() + "\n\n"

def format_one_file(item):
    return (
        f"FILE: {item['filename']}\n"
        f"{item['content']}\n"
        f"{'-'*60}\n"
    )

def build_prompt_with_limit(pipeline_contents, max_input_tokens=90000):
    """
    Build prompt incrementally and stop before token limit.
    Returns:
    - prompt text
    - included file records
    """
    base_prompt = build_base_prompt()
    current_prompt = base_prompt
    included_files = []

    for item in pipeline_contents:
        file_block = format_one_file(item)
        candidate_prompt = current_prompt + file_block

        if rough_token_estimate(candidate_prompt) > max_input_tokens:
            break

        current_prompt = candidate_prompt
        included_files.append(item)

    return current_prompt, included_files

def copy_used_prompt_files(included_files, output_dir):
    """
    Copy the files actually used in the prompt into a new directory.
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    copied = []
    for item in included_files:
        src = Path(item["path"])
        dst = output_path / item["filename"]
        shutil.copy2(src, dst)
        copied.append(str(dst))

    return copied

def identify_patterns(pipeline_contents, used_files_output_dir):
    llm = ChatOpenAI(
        model=MODEL_NAME,
        api_key=OPENAI_API_KEY,
        temperature=0,
        max_tokens=MAX_OUTPUT_TOKENS
    )

    prompt, included_files = build_prompt_with_limit(
        pipeline_contents,
        max_input_tokens=MAX_INPUT_TOKENS
    )

    print(f"Included files: {len(included_files)} / {len(pipeline_contents)}")
    print(f"Estimated prompt tokens: {rough_token_estimate(prompt)}")

    copied_files = copy_used_prompt_files(included_files, used_files_output_dir)
    print(f"Copied {len(copied_files)} prompted files to: {used_files_output_dir}")

    response = llm.invoke(prompt)
    return response.content, included_files, copied_files

def main(directory, start_idx=None, end_idx=None):
    print("Reading pipeline files...")
    pipeline_contents = read_pipeline_files(directory, start_idx, end_idx)
    print(f"Total files read: {len(pipeline_contents)}")
    print(f"Requested index range: [{start_idx}:{end_idx}]")

    # Optional: sort smaller files first to fit more files
    pipeline_contents.sort(key=lambda x: len(x["content"]))

    used_files_output_dir = os.path.join(
        "/Users/ankita/Desktop/", #Input directory path to be created
        f"llm_prompt_used_files_{start_idx}_{end_idx}"
    )

    print("Identifying patterns...")
    patterns, included_files, copied_files = identify_patterns(
        pipeline_contents,
        used_files_output_dir
    )

    print("\nPatterns Identified:")
    print(patterns)

    print("\nFiles actually included in prompt:")
    for item in included_files:
        print(item["filename"])

    print("\nCopied files directory:")
    print(used_files_output_dir)

if __name__ == "__main__":
    main("/Users/ankita/Desktop/V3_DatasetTest_Valid",0, 29069) #Put your dataset path here

Reading pipeline files...
Total files read: 29069
Requested index range: [0:29069]
Identifying patterns...
Included files: 1056 / 29069
Estimated prompt tokens: 89999
Copied 1056 prompted files to: /Users/ankita/Desktop/llm_prompt_used_files_0_29069

Patterns Identified:
Based on the analysis of the provided DevOps YAML pipeline files, the following recurring structural patterns have been identified:

### 1. Basic CI/CD Pipeline Structure
This pattern is characterized by a simple structure that includes a job that runs on a specific operating system, checks out the code, and executes a command. This structure is very common across many pipelines.

```yaml
name: <name>
on: [push]
jobs:
  <job_name>:
    runs-on: <os>
    steps:
    - name: "Checkout code"
      uses: actions/checkout@<version>
    - name: "<step_name>"
      run: "<command>"
```

### 2. Linting and Testing Pipeline
This pattern includes a job that runs linting and testing commands after checking out the code. It is ofte